In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

In [ ]:
print("PyMC version:", pm.__version__)

In [ ]:
data = fetch_california_housing()

In [ ]:
print(data.keys())

In [ ]:
print(data.data[:5])

In [ ]:
print(data.target[:5])

In [ ]:
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [ ]:
df = X.copy()
df["price"] = y

In [ ]:
print(df.head())

In [ ]:
df.describe()

In [ ]:
sns.set(style="whitegrid")

# Target distribution
plt.figure(figsize=(8,5))
sns.histplot(df['price'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of House Prices (Target)')
plt.xlabel('Price (in $100k)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Feature distributions
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']
df[features].hist(bins=30, figsize=(12,8), color='lightgreen')
plt.suptitle('Feature Distributions', fontsize=16)
plt.show()

In [ ]:
# Scatterplots: feature vs target
for feat in features:
    plt.figure(figsize=(6,4))
    sns.scatterplot(x=df[feat], y=df['price'])
    plt.title(f'{feat} vs Price')
    plt.xlabel(feat)
    plt.ylabel('Price')
    plt.show()

In [ ]:
# Pairplot of key features with target
key_features = ['MedInc', 'HouseAge', 'AveRooms', 'price']
sns.pairplot(df[key_features], diag_kind='kde', corner=True)
plt.suptitle('Pairplot of Key Features vs Price', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix', fontsize=16)
plt.show()

In [ ]:
# Boxplots to detect outliers
plt.figure(figsize=(12,6))
sns.boxplot(data=df[features], palette='pastel')
plt.title('Boxplots of Features (Outlier Detection)')
plt.show()

In [ ]:
# Geographic scatterplot (latitude & longitude)
plt.figure(figsize=(8,6))
sns.scatterplot(x='Longitude', y='Latitude', hue='price', data=df, palette='viridis', legend=False)
plt.title('House Prices by Location')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
# Select features
selected_features = ['MedInc', 'HouseAge', 'AveRooms', 'Latitude']
X = df[selected_features]
y = df['price']

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Check shapes
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# Scale features (fit scaler on train set only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Check shapes
print(X_train_scaled.shape, X_test_scaled.shape, y_train.shape, y_test.shape)

In [ ]:
with pm.Model() as housing_model:
    # Containers for data (allows for easy out-of-sample prediction later)
    # In PyMC 5.x, pm.Data is mutable by default
    X_shared = pm.Data("X_shared", X_train_scaled)
    y_shared = pm.Data("y_shared", y_train)

    # --- Priors ---
    # Intercept: Expecting price around the mean (0 after scaling, but y isn't scaled here)
    intercept = pm.Normal("intercept", mu=y_train.mean(), sigma=1)
    
    # Coefficients: One for each feature
    beta = pm.Normal("beta", mu=0, sigma=1, shape=X_train_scaled.shape[1])
    
    # Error term (Standard Deviation of the residuals)
    sigma = pm.HalfNormal("sigma", sigma=1)

    # --- Likelihood ---
    # Linear equation: y = alpha + X * beta
    mu = intercept + pm.math.dot(X_shared, beta)
    
    # Observe the data
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y_shared)

1. Sample the posterior
2. Inspect sampling diagnostics
3. Posterior summary
4. Posterior predictive check
5. Out-of-sample prediction
6. Model criticism / evaluation

In [ ]:
with housing_model:
    trace = pm.sample(
        draws=2000,
        tune=1000,
        target_accept=0.9,
        return_inferencedata=True,
        idata_kwargs={"log_likelihood": True}
    )

In [ ]:
# Inspect sampling diagnostics
import arviz as az

az.plot_trace(trace)
az.summary(trace)

In [ ]:
# Posterior summary
az.summary(trace, var_names=["intercept", "beta", "sigma"])

In [ ]:
# Posterior predictive check
with housing_model:
    ppc = pm.sample_posterior_predictive(trace)

az.plot_ppc(ppc)

In [ ]:
# Out-of-sample prediction
with housing_model:
    pm.set_data({
        "X_shared": X_test_scaled,
        "y_shared": np.zeros(len(X_test_scaled))   # placeholder so shapes match
    })

    preds = pm.sample_posterior_predictive(trace)

# Posterior predictive mean
y_pred = preds.posterior_predictive["y_obs"].mean(dim=("chain", "draw"))

# Evaluate model
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred)

print("Test R²:", r2)

In [ ]:
# Check specific stats
import numpy as np

y_rep = preds.posterior_predictive["y_obs"].values

print("Real mean:", np.mean(y_test))
print("Simulated mean:", y_rep.mean())

print("Real std:", np.std(y_test))
print("Simulated std:", y_rep.std())

In [ ]:
# Residual Analysis
residuals = y_test - y_pred

plt.scatter(y_pred, residuals)
plt.axhline(0, color='red')

In [ ]:
import scipy.stats as stats
stats.probplot(residuals, dist="norm", plot=plt)

In [ ]:
y_samples = preds.posterior_predictive["y_obs"]

# 95% interval coverage
lower = np.percentile(y_samples, 2.5, axis=(0,1))
upper = np.percentile(y_samples, 97.5, axis=(0,1))

coverage = np.mean((y_test >= lower) & (y_test <= upper))
print("Coverage:", coverage)

In [ ]:
az.loo(trace)

In [ ]:
# Model criticism / evaluation
# Posterior Predictive vs Actual

plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Predicted vs Actual House Prices")

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--'
)

plt.show()

In [ ]:
# Residual Analysis

residuals = y_test - y_pred

plt.figure(figsize=(6,4))
plt.hist(residuals, bins=50)
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Feature Coefficient Interpretation

az.plot_forest(trace, var_names=["beta"], combined=True)
plt.title("Posterior of Regression Coefficients")

In [ ]:
# HDI intervals
az.plot_posterior(trace, var_names=["beta"])

# R-hat convergence
az.summary(trace, round_to=2)